# inplace-param-update — worked example 3: no_grad block lets you mutate the param directly

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-param-update`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Inside a `torch.no_grad()` block you can write `param -= lr * grad` directly on an `nn.Parameter` — the operation is in place and untracked, so it both updates the model's storage and avoids polluting the autograd graph. This is equivalent to using `param.data` but reads more naturally.

## Worked solution

We do a full mini training loop step the way it is usually written inside a real optimizer's `step()`.

1. **Forward + backward.** Run the layer on a fixed input, take a sum-of-squares loss, and call `loss.backward()` to populate `p.grad` for each parameter.
2. **Enter no_grad.** `with t.no_grad():` ensures the parameter updates are not recorded as graph operations — otherwise we would build an ever-growing graph across steps.
3. **In-place update.** For each parameter, `p -= lr * p.grad`. On an `nn.Parameter` inside `no_grad`, the `-=` is a genuine in-place mutation of the existing storage, so `data_ptr()` is preserved.
4. **Zero the grads.** `p.grad.zero_()` resets accumulation for the next step, again in place.

The demo confirms the parameter pointer is stable across the step and the loss strictly decreases after applying the update.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)

lin = nn.Linear(5, 1)
x = t.randn(8, 5)
target = t.zeros(8, 1)

def loss_of(model):
    return ((model(x) - target) ** 2).sum()

l_before = loss_of(lin)
l_before.backward()
ptr0 = lin.weight.data_ptr()
with t.no_grad():
    for p in lin.parameters():
        p -= 0.01 * p.grad   # in-place, untracked
        p.grad.zero_()
l_after = loss_of(lin)

print('weight ptr stable:', lin.weight.data_ptr() == ptr0)
print('loss decreased:', l_after.item() < l_before.item())